# Phase 1: 4H-SiC Material Parameters and Device Electrostatics

This notebook validates the complete Phase 1 simulation pipeline:
1. Material parameters for 4H-SiC
2. Incomplete ionization of Al acceptors
3. Analytical electrostatics (built-in potential, depletion width, E-field)
4. Numerical devsim Poisson solver
5. Comparison against experimental C-V data

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.sic_material import SiC4H_Parameters, compute_ni, mobility_caughey_thomas
from src.incomplete_ionization import ionized_acceptor_fraction, ionized_acceptor_concentration
from src.analytical import built_in_potential, depletion_width, electric_field_profile, depletion_width_vs_bias
from src.device import create_sic_device, DEFAULT_N_D
from src.poisson import (
    setup_poisson, solve_equilibrium, extract_electric_field,
    extract_depletion_width, extract_depletion_width_numerical,
    voltage_sweep
)
from src.plotting import (
    plot_electric_field, plot_electric_field_multi,
    plot_depletion_width_vs_bias, plot_doping_profile, save_figure
)

params = SiC4H_Parameters()
print('=== 4H-SiC Material Parameters ===')
for field in ['E_g', 'eps_r', 'n_i_300', 'mu_n_max', 'mu_p_max',
              'tau_n', 'tau_p', 'C_n', 'C_p', 'E_A', 'g_A']:
    print(f'  {field:12s} = {getattr(params, field)}')

In [ ]:
# Cell 2: Incomplete Ionization
N_A = 1e19  # Al acceptor concentration
f_ion = ionized_acceptor_fraction(N_A, T=300)
N_A_ionized = ionized_acceptor_concentration(N_A, T=300)

print(f'N_A = {N_A:.1e} cm^-3')
print(f'Ionization fraction = {f_ion*100:.1f}%')
print(f'N_A^- = {N_A_ionized:.2e} cm^-3')

# Plot ionization fraction vs N_A
N_A_range = np.logspace(15, 20, 100)
f_range = [ionized_acceptor_fraction(na, T=300) for na in N_A_range]

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(N_A_range, np.array(f_range)*100)
ax.axhline(y=f_ion*100, color='r', linestyle='--', alpha=0.5)
ax.axvline(x=N_A, color='r', linestyle='--', alpha=0.5)
ax.set_xlabel('Total Acceptor Concentration $N_A$ (cm$^{-3}$)')
ax.set_ylabel('Ionization Fraction (%)')
ax.set_title('Al Acceptor Incomplete Ionization in 4H-SiC at 300K')
ax.set_ylim([0, 100])
ax.grid(True, alpha=0.3)
ax.annotate(f'N_A=1e19: {f_ion*100:.1f}%', xy=(N_A, f_ion*100),
            xytext=(1e17, f_ion*100+15), fontsize=11,
            arrowprops=dict(arrowstyle='->', color='red'))
plt.show()

In [ ]:
# Cell 3: Built-in Potential
N_D = DEFAULT_N_D
n_i = params.n_i_300

V_bi = built_in_potential(N_A_ionized, N_D, n_i)
print(f'Built-in Potential Calculation:')
print(f'  N_A^- = {N_A_ionized:.2e} cm^-3')
print(f'  N_D   = {N_D:.2e} cm^-3')
print(f'  n_i   = {n_i:.2e} cm^-3')
print(f'  V_bi  = {V_bi:.3f} V')
print(f'\nComponents: kT*ln(N_A^-*N_D/n_i^2)')
print(f'  kT = {0.02585:.5f} eV')
print(f'  ln(N_A^-*N_D/n_i^2) = {np.log(N_A_ionized*N_D/n_i**2):.2f}')

In [ ]:
# Cell 4: Analytical Depletion Width vs Bias
epi_thickness = 10e-4  # 10 um in cm
V_biases = np.linspace(0, -60, 200)

W_ana = depletion_width_vs_bias(V_biases, V_bi, N_D, eps_r=params.eps_r,
                                 epi_thickness=epi_thickness)

# Experimental data
V_exp = [0, -10, -30]
W_exp = [1.7e-4, 9.5e-4, 9.73e-4]  # cm

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(V_biases, W_ana * 1e4, 'b-', linewidth=2, label='Analytical')
ax.axhline(y=epi_thickness*1e4, color='gray', linestyle=':', label='Epi thickness')
ax.plot(V_exp, np.array(W_exp)*1e4, 'ro', markersize=8, label='Experimental (C-V)')
ax.set_xlabel('Applied Bias (V)')
ax.set_ylabel('Depletion Width ($\\mu$m)')
ax.set_title('Depletion Width vs Reverse Bias')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 12])
plt.show()

print('\nW at key voltages:')
for V in [0, -5, -10, -30, -60]:
    W = depletion_width(V_bi, V, N_D, eps_r=params.eps_r, epi_thickness=epi_thickness)
    print(f'  W({V:3d}V) = {W*1e4:.2f} um')

In [ ]:
# Cell 5: Analytical E-field Profiles
x_cm = np.linspace(0, epi_thickness, 1000)

fig, ax = plt.subplots(figsize=(8, 5))
for V, color in [(0, 'blue'), (-5, 'green'), (-10, 'orange'), (-30, 'red')]:
    E = electric_field_profile(x_cm, V_bi, V, N_D, eps_r=params.eps_r)
    ax.plot(x_cm*1e4, E, color=color, label=f'{V}V')

ax.set_xlabel('Depth from Junction ($\\mu$m)')
ax.set_ylabel('Electric Field (V/cm)')
ax.set_title('Analytical E-field Profiles at Multiple Biases')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Cell 6: devsim Numerical Solution
print('Creating devsim device...')
dev = create_sic_device()
print(f'Device: {dev["num_nodes"]} nodes, junction at {dev["junction_pos"]*1e4:.1f} um')
print(f'N_A_ionized = {dev["N_A_ionized"]:.2e}, N_D = {dev["N_D"]:.2e}')

# Solve equilibrium
print('\nSolving Poisson equation...')
setup_poisson(dev)
solve_equilibrium(dev)

# Extract equilibrium E-field
x_eq, E_eq = extract_electric_field(dev)
W_num_0V = extract_depletion_width_numerical(dev)
W_ana_0V = extract_depletion_width(dev, V_applied=0.0)

print(f'\nEquilibrium results:')
print(f'  E_max = {np.max(np.abs(E_eq)):.2e} V/cm')
print(f'  W(0V) numerical  = {W_num_0V*1e4:.2f} um')
print(f'  W(0V) analytical = {W_ana_0V*1e4:.2f} um')
print(f'  Agreement: {abs(W_num_0V-W_ana_0V)/W_ana_0V*100:.1f}%')

# Plot equilibrium E-field
fig, ax = plt.subplots(figsize=(8, 5))
plot_electric_field(x_eq, E_eq, voltage_label='0V (equilibrium)', ax=ax)
plt.show()

In [ ]:
# Cell 7: Numerical vs Analytical Comparison

# Depletion width comparison
V_range = np.linspace(0, -60, 100)
W_ana_arr = np.array([extract_depletion_width(dev, V) for V in V_range])
W_pure_ana = depletion_width_vs_bias(V_range, V_bi, N_D, eps_r=params.eps_r,
                                      epi_thickness=epi_thickness)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Analytical E-field vs numerical at 0V
x_ana = np.linspace(0, epi_thickness, 500)
E_ana = electric_field_profile(x_ana, V_bi, 0, N_D, eps_r=params.eps_r)
jp = dev['junction_pos']
ax1.plot((x_eq - jp)*1e4, E_eq, 'b-', linewidth=2, label='Numerical (devsim)')
ax1.plot(x_ana*1e4, E_ana, 'r--', linewidth=1.5, label='Analytical')
ax1.set_xlabel('Distance from Junction ($\\mu$m)')
ax1.set_ylabel('Electric Field (V/cm)')
ax1.set_title('E-field at 0V: Numerical vs Analytical')
ax1.set_xlim([-0.5, 5])
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: Depletion width comparison
plot_depletion_width_vs_bias(V_range, W_ana_arr, W_analytical=W_pure_ana, ax=ax2)
ax2.set_title('Depletion Width: Simulation vs Analytical vs Experimental')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Validation Summary Table
print('=' * 75)
print('PHASE 1 VALIDATION SUMMARY')
print('=' * 75)
print()
print(f'{"Voltage":>8s} | {"W_numerical":>12s} | {"W_analytical":>12s} | {"W_experimental":>14s} | {"Error vs Exp":>12s}')
print('-' * 75)

exp_data = {0: 1.7, -10: 9.5, -30: 9.73}  # um

for V in [0, -10, -30]:
    W_sim = extract_depletion_width(dev, V_applied=V) * 1e4  # um
    W_ana_val = depletion_width(V_bi, V, N_D, eps_r=params.eps_r,
                                epi_thickness=epi_thickness) * 1e4
    W_exp_val = exp_data[V]
    error = abs(W_sim - W_exp_val) / W_exp_val * 100
    
    print(f'{V:8d}V | {W_sim:10.2f} um | {W_ana_val:10.2f} um | {W_exp_val:12.2f} um | {error:10.1f}%')

print()
print('Notes:')
print('  - W(0V) matches well (calibrated N_D to this target)')
print('  - W(-10V) and W(-30V) discrepancy is due to single-N_D model limitation')
print('  - Experimental device likely has non-uniform doping or graded junction')
print('  - Punch-through behavior (W saturation) correctly captured')
print()
print(f'Key parameters:')
print(f'  N_D (calibrated) = {N_D:.2e} cm^-3')
print(f'  N_A_ionized      = {N_A_ionized:.2e} cm^-3 ({f_ion*100:.1f}% of N_A={N_A:.0e})')
print(f'  V_bi             = {V_bi:.3f} V')
print(f'  n_i              = {n_i:.2e} cm^-3')
print(f'  Epi thickness    = {epi_thickness*1e4:.0f} um')